# Pre-train the multigauge models on CAMELS-US

This notebook builds the pre-trained models that `multigauge` fine-tunes on individual basins:

1. an **LSTM** with a probabilistic (CMAL) output, trained on the 531 CAMELS-US basins used by most deep-learning studies;
2. optionally, the **δHBV hybrid** (differentiable HBV with neural parameter learning).

**Before you start:** *Runtime → Change runtime type → GPU* (a free T4 is enough). Expect about 1 hour for the LSTM and 20–30 minutes for the hybrid. The CAMELS download is about 3.2 GB.

Data: Newman et al. (2015) and Addor et al. (2017), CAMELS, NCAR GDEX (https://gdex.ucar.edu/dataset/camels.html).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Install multigauge

In [ ]:
!git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git
!pip install -q -e multigauge-hydro

## 2. Download CAMELS-US (forcings + streamflow, and catchment areas)

In [ ]:
%%bash
mkdir -p /content/camels && cd /content/camels
BASE=https://gdex.ucar.edu/dataset/camels/file
[ -f basin_timeseries_v1p2_metForcing_obsFlow.zip ] || wget -q --show-progress $BASE/basin_timeseries_v1p2_metForcing_obsFlow.zip
[ -f camels_topo.txt ] || wget -q $BASE/camels_topo.txt
[ -d basin_dataset_public_v1p2 ] || unzip -q basin_timeseries_v1p2_metForcing_obsFlow.zip
ls

## 3. Load the 531 basins

In [ ]:
import numpy as np, pandas as pd, multigauge as mg
from multigauge.camels import load_camels
ids = open("multigauge-hydro/notebooks/camels_531_basins.txt").read().split()
basins = load_camels("/content/camels", ids, progress=lambda i, n, b: print(f"\r{i}/{n}", end=""))
print("\nloaded", len(basins), "basins")

## 4. Pre-train the LSTM

Standard CAMELS periods (Kratzert et al., 2019): train 1980-10-01 to 1995-09-30, validate 1995-10-01 to 2000-09-30, test 2000-10-01 to 2010-09-30.

In [ ]:
from multigauge.nn import ModelConfig
from multigauge.transfer import pretrain
cfg = ModelConfig.preset("pretrain")   # 128 hidden units, 365-day lookback, CMAL output
print(cfg)
model = pretrain(basins, cfg, ("1980-10-01", "1995-09-30"), ("1995-10-01", "2000-09-30"),
                 progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
model.info.update(source="CAMELS-US", description="LSTM pre-trained on 531 CAMELS-US basins (Daymet forcings, CMAL output).")
model.save("multigauge-hydro/src/multigauge/pretrained/camels-us")

## 5. Test-period skill (2000-10-01 to 2010-09-30)

In [ ]:
from multigauge.transfer import period_mask
model.net.to("cuda" if torch.cuda.is_available() else "cpu")
rows = []
for b in basins:
    m = period_mask(b.dates, ("2000-10-01", "2010-09-30"))
    p = model.predict(b)
    rows.append({"basin": b.basin_id, **mg.evaluate_probabilistic(b.q_mm.to_numpy()[m], p[m])})
model.net.to("cpu")
test = pd.DataFrame(rows).set_index("basin")
test.to_csv("multigauge-hydro/src/multigauge/pretrained/camels-us/test_metrics.csv")
print(test[["NSE", "KGE", "CRPS", "Coverage90"]].median().round(3))

*For reference:* published single-forcing CAMELS LSTMs reach a median test NSE of roughly 0.72–0.74. This model uses only climate attributes computed from the forcings (so it can run on any U.S. basin without extra attribute data), so a slightly lower median is expected.

## 6. (Optional) Pre-train the δHBV hybrid

In [ ]:
from multigauge.hybrid import HybridConfig, HybridModel
tr = [period_mask(b.dates, ("1980-10-01", "1995-09-30")) for b in basins]
va = [period_mask(b.dates, ("1995-10-01", "2000-09-30")) for b in basins]
hyb = HybridModel(HybridConfig(epochs=30)).fit(basins, tr, va,
        progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
hyb.save("multigauge-hydro/src/multigauge/pretrained/camels-us-dhbv")

## 7. Download the trained models

Attach `multigauge_pretrained.zip` in your chat with Claude (or commit the folders under `src/multigauge/pretrained/`).

In [ ]:
!cd multigauge-hydro/src/multigauge && zip -qr /content/multigauge_pretrained.zip pretrained/camels-us pretrained/camels-us-dhbv 2>/dev/null; ls -la /content/multigauge_pretrained.zip
from google.colab import files
files.download("/content/multigauge_pretrained.zip")